In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [27]:
'''

FIGURE 2
'''

# Load metadata
path = Path("data/PilotDatasetSubjects.xlsx")
df = pd.read_excel(path, sheet_name=0, skiprows=1)

# Basic cleaning
df = df.copy()
# -----------------------------
# Basic cleaning
# -----------------------------
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()

# -----------------------------
# Basic cleaning
# -----------------------------
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()
df["Education"] = df["Education"].astype(str).str.strip()

gender_map = {"F": "Female", "M": "Male"}
df["GenderLabel"] = df["Gender"].map(gender_map).fillna("Other/Unknown")

# Remove fake nan strings if present
df["Education"] = df["Education"].replace(["nan", "NaN", "None", ""], np.nan)

# -----------------------------
# Basic statistics
# -----------------------------
age = df["Age"].dropna()
mean_age = age.mean()
median_age = age.median()
min_age = age.min()
max_age = age.max()

gender_counts = df["GenderLabel"].value_counts()
female_n = int(gender_counts.get("Female", 0))
male_n = int(gender_counts.get("Male", 0))
other_n = int(gender_counts.get("Other/Unknown", 0))

education_counts = (
    df["Education"]
    .dropna()
    .value_counts()
    .sort_values(ascending=False)
)

# -----------------------------
# Style
# -----------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

blue = "#123B7A"
red = "#C62828"
gray = "#6E6E6E"

# -----------------------------
# Helper to force visible axes
# -----------------------------
def style_axis(ax):
    ax.set_facecolor("white")
    ax.tick_params(axis="both", colors="black", labelcolor="black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.title.set_color("black")
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# -----------------------------
# Panel A: Age histogram
# -----------------------------
ax = axes[0]
bins = np.arange(np.floor(min_age / 5) * 5, np.ceil(max_age / 5) * 5 + 5, 5)

ax.hist(age, bins=bins, color=blue, edgecolor="white")
ax.axvline(mean_age, color=red, linestyle="--", linewidth=2, label=f"Mean = {mean_age:.1f}")
ax.axvline(median_age, color=gray, linestyle=":", linewidth=2, label=f"Median = {median_age:.1f}")

ax.set_title("A. Age distribution", loc="left", fontweight="bold")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.legend(frameon=False)

style_axis(ax)

# -----------------------------
# Panel B: Gender bar plot
# -----------------------------
ax = axes[1]

plot_labels = []
plot_counts = []
plot_colors = []

if female_n > 0:
    plot_labels.append("Female")
    plot_counts.append(female_n)
    plot_colors.append(red)

if male_n > 0:
    plot_labels.append("Male")
    plot_counts.append(male_n)
    plot_colors.append(blue)

if other_n > 0:
    plot_labels.append("Other/Unknown")
    plot_counts.append(other_n)
    plot_colors.append(gray)

bars = ax.bar(plot_labels, plot_counts, color=plot_colors, edgecolor="black", linewidth=0.8)

ax.set_title("B. Gender distribution", loc="left", fontweight="bold")
ax.set_xlabel("Gender")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")

for bar, count in zip(bars, plot_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{count}",
        ha="center",
        va="bottom",
        fontsize=11,
        color="black"
    )

style_axis(ax)

# -----------------------------
# Panel C: Education level bar plot
# -----------------------------
ax = axes[2]

edu_labels = education_counts.index.tolist()
edu_values = education_counts.values.tolist()

bars = ax.bar(edu_labels, edu_values, color=blue, edgecolor="black", linewidth=0.8)

ax.set_title("C. Education level", loc="left", fontweight="bold")
ax.set_xlabel("Education")
ax.set_ylabel("Number of participants")
ax.grid(True, axis="y", alpha=0.25, color="0.75")
ax.tick_params(axis="x", rotation=30)

for bar, count in zip(bars, edu_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{count}",
        ha="center",
        va="bottom",
        fontsize=10,
        color="black"
    )

style_axis(ax)

# -----------------------------
# Panel D: Age by gender boxplot
# -----------------------------
ax = axes[3]

box_data = []
tick_labels = []
box_colors = []

for label, color in [("Female", red), ("Male", blue), ("Other/Unknown", gray)]:
    vals = df.loc[df["GenderLabel"] == label, "Age"].dropna().values
    if len(vals) > 0:
        box_data.append(vals)
        tick_labels.append(label)
        box_colors.append(color)

bp = ax.boxplot(
    box_data,
    tick_labels=tick_labels,
    patch_artist=True,
    widths=0.45
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.45)
    patch.set_edgecolor("black")

for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(2)

for whisker in bp["whiskers"]:
    whisker.set_color("black")

for cap in bp["caps"]:
    cap.set_color("black")

for flier in bp["fliers"]:
    flier.set_markeredgecolor("black")
    flier.set_markerfacecolor("black")
    flier.set_alpha(0.5)

rng = np.random.default_rng(42)
for i, vals in enumerate(box_data, start=1):
    x_jitter = rng.normal(i, 0.05, size=len(vals))
    ax.scatter(x_jitter, vals, s=18, alpha=0.5, color="black")

ax.set_title("D. Age by gender", loc="left", fontweight="bold")
ax.set_xlabel("Gender")
ax.set_ylabel("Age (years)")
ax.grid(True, axis="y", alpha=0.25, color="0.75")

style_axis(ax)

# -----------------------------
# Final layout
# -----------------------------
fig.suptitle("Pilot dataset metadata overview", fontsize=17, fontweight="bold", color="black")
plt.tight_layout()
plt.subplots_adjust(top=0.92)

# -----------------------------
# Save
# -----------------------------
pdf_path = "MainFigures/F2/pilot_dataset_metadata_multipanel_fixed.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close()

In [13]:
'''

FIGURE 3
'''
# Load data
path = "FindOptimalProbForAugmentation/contrastive_optuna_results_old.csv"
df = pd.read_csv(path)

# Clean channel names for display
df["ChannelShort"] = df["Channel"].str.replace("EEG.", "", regex=False)

# Best = minimum objective
best_idx = df.groupby("Channel")["Score"].idxmin()
best_df = (
    df.loc[best_idx, [
        "Channel", "ChannelShort", "Trial", "Score", "HiddenSize", "ProjSize", "Temp",
        "KernelSize", "Padding", "Stride", "KernelSize2", "Padding2", "Stride2"
    ]]
    .sort_values("ChannelShort")
    .reset_index(drop=True)
)

channel_order = best_df["Channel"].tolist()

# Explicit white background
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

fig, axes = plt.subplots(4, 4, figsize=(18, 14), constrained_layout=True, facecolor="white")
axes = axes.flatten()

line_color = "#123B7A"   # dark blue
dot_color = "#C62828"    # red

# Channel panels
for i, channel in enumerate(channel_order):
    ax = axes[i]
    sub = df[df["Channel"] == channel].sort_values("Trial").copy()
    sub["cum_best"] = sub["Score"].cummin()

    # Trial points in red
    ax.scatter(sub["Trial"], sub["Score"], alpha=0.8, s=20, color=dot_color)
    # Running best in dark blue
    ax.plot(sub["Trial"], sub["cum_best"], linewidth=2.2, color=line_color)

    best_row = sub.loc[sub["Score"].idxmin()]
    ax.scatter([best_row["Trial"]], [best_row["Score"]], s=110, marker="*", color=dot_color)

    ax.set_title(channel.replace("EEG.", ""), color="black")
    ax.set_xlabel("Trial", color="black")
    ax.set_ylabel("Objective", color="black")
    ax.grid(True, alpha=0.25, color="0.7")
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

    txt = (
        f"H={int(best_row['HiddenSize'])}, P={int(best_row['ProjSize'])}\n"
        f"K1={int(best_row['KernelSize'])}, K2={int(best_row['KernelSize2'])}\n"
        f"T={best_row['Temp']:.3f}"
    )
    ax.text(
        0.03, 0.03, txt,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        color="black",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.9)
    )



# Save outputs
pdf_path = "MainFigures/F3/contrastive_multipanel_figure.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
plt.close(fig)



In [20]:
'''

FIGURE 3
'''
# Basic cleanup
df["ChannelShort"] = df["Channel"].astype(str).str.replace("EEG.", "", regex=False)

# Best trial per channel based on minimization objective
best_idx = df.groupby("Channel")["Score"].idxmin()
best_df = (
    df.loc[best_idx, [
        "Channel", "ChannelShort", "Trial", "Score", "Alignment", "Uniformity",
        "HiddenSize", "ProjSize", "Temp", "KernelSize", "KernelSize2"
    ]]
    .sort_values("ChannelShort")
    .reset_index(drop=True)
)

channel_order = best_df["Channel"].tolist()

# Global axis limits for comparability across panels
align_min, align_max = df["Alignment"].min(), df["Alignment"].max()
uni_min, uni_max = df["Uniformity"].min(), df["Uniformity"].max()

# Add small margins
align_pad = (align_max - align_min) * 0.05 if align_max > align_min else 0.05
uni_pad = (uni_max - uni_min) * 0.05 if uni_max > uni_min else 0.05

# White figure style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.size": 10,
})

fig, axes = plt.subplots(4, 4, figsize=(18, 14), constrained_layout=True, facecolor="white")
axes = axes.flatten()

line_color = "#123B7A"   # dark blue
dot_color = "#C62828"    # red

# 14 panels: alignment vs uniformity for each channel
for i, channel in enumerate(channel_order):
    ax = axes[i]
    sub = df[df["Channel"] == channel].copy()
    best_row = sub.loc[sub["Score"].idxmin()]

    # All trials
    ax.scatter(
        sub["Alignment"],
        sub["Uniformity"],
        s=26,
        alpha=0.8,
        color=dot_color
    )

    # Best trial
    ax.scatter(
        [best_row["Alignment"]],
        [best_row["Uniformity"]],
        s=130,
        marker="*",
        color=line_color,
        zorder=5
    )

    # Optional guiding lines through best point
    ax.axvline(best_row["Alignment"], color=line_color, alpha=0.15, linewidth=1.2)
    ax.axhline(best_row["Uniformity"], color=line_color, alpha=0.15, linewidth=1.2)

    ax.set_title(channel.replace("EEG.", ""), color="black")
    ax.set_xlabel("Alignment", color="black")
    ax.set_ylabel("Uniformity", color="black")
    ax.set_xlim(align_min - align_pad, align_max + align_pad)
    ax.set_ylim(uni_min - uni_pad, uni_max + uni_pad)
    ax.grid(True, alpha=0.25, color="0.7")
    ax.tick_params(colors="black")
    for spine in ax.spines.values():
        spine.set_color("black")

    txt = (
        f"best trial={int(best_row['Trial'])}\n"
        f"A={best_row['Alignment']:.3f}\n"
        f"U={best_row['Uniformity']:.3f}\n"
        f"obj={best_row['Score']:.3f}\n"
        f"K1={int(best_row['KernelSize'])}, K2={int(best_row['KernelSize2'])}"
    )
    ax.text(
        0.03, 0.03, txt,
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        ha="left",
        color="black",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.9)
    )

# Summary panel 15: best-trial alignment by channel
ax = axes[14]
align_plot = best_df.sort_values("Alignment", ascending=True)
ax.bar(align_plot["ChannelShort"], align_plot["Alignment"], color=line_color)
ax.set_title("Best-trial alignment by channel", color="black")
ax.set_xlabel("Channel", color="black")
ax.set_ylabel("Alignment", color="black")
ax.tick_params(axis="x", rotation=45, colors="black")
ax.tick_params(axis="y", colors="black")
ax.grid(True, axis="y", alpha=0.25, color="0.7")
for spine in ax.spines.values():
    spine.set_color("black")

# Summary panel 16: best-trial uniformity by channel
ax = axes[15]
uni_plot = best_df.sort_values("Uniformity", ascending=True)
ax.bar(uni_plot["ChannelShort"], uni_plot["Uniformity"], color=dot_color)
ax.set_title("Best-trial uniformity by channel", color="black")
ax.set_xlabel("Channel", color="black")
ax.set_ylabel("Uniformity", color="black")
ax.tick_params(axis="x", rotation=45, colors="black")
ax.tick_params(axis="y", colors="black")
ax.grid(True, axis="y", alpha=0.25, color="0.7")
for spine in ax.spines.values():
    spine.set_color("black")

fig.suptitle(
    "Alignment and uniformity across EEG channels\nRed = all trials, blue star = best trial by minimum objective",
    fontsize=16,
    color="black"
)

pdf_path = "MainFigures/F3/contrastive_alignment_uniformity_multipanel.pdf"
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.close(fig)



KeyError: 'Channel'